# Neural Networks for image and tabular data

* Go over basics of NNs in general
* Go over Torch
* Introduce dense layers
* Introdice convolusional layers
* Do the jet-tagger task
* Do the cosmo-classifer task

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# PyTorch (for deep learning)
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary

# fits
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.visualization import simple_norm

# sklearn (for machine learning)
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_blobs

np.random.seed(42) # For reproducibility, we set the "seed" of our random number generator to a fixed number

# Pytorch

Pytorch is a package used to build neural networks.

In `Pytorch` (refered to as `torch` in code), we can build a model by creating a `Class`, and adding different layer objects within the `__init__` block of the class to define the architecture. The first layer we add is the input layer, and then the last layer we add is our output layer.

You'll also notice that aside from defining the structure of the model, we also define a function called `forward`, which is the function that will be called when we pass data through the model. In this case, we just pass data from one layer to another in a series. In some other cases, you might want to define a more complex `forward` function that does something more complicated with the input data.

Think of the model like a sandwich - it always has bread on the top and the bottom and the contents of the sandwich can vary. 
The "bread" in the model is the input and output layers. 
The input layers need to take the features of our dataset, our x values, and translate them in to a format the rest of the model can use. 
The output layers need to take the next to last layer of the model and transform that output into our "target" variables, the y values we can use to evaluate a loss function. 

Our model sandwich below is pretty boring, an input layer, a single learned transformation, and an output layer.

The dense layers here are simple, they are a matrix of coefficients that modify the input. 
Those coefficients are trainable, they're the parameters of our model. 
(Like how m and b are the parameters of y = m*x+b)

The [`ReLU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html#relu)("Rectified Linear Unit") layers are an "Activation" function, they change the distribution out of the dense output from a linear output to something slightly more non-linear.
They can improve the output of the model by increasing how much non-linearity the model can express. 

Pytorch has a few main pieces. 

## The Model

Each model is defined as a subclass of [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module). This is how models can be trained (have a way to update the weights automatically), calculate and apply gradients according to loss functions, and use advanced features like sending parameters to specific devices like cpu and gpu.

A basic model looks like this: 

```python 
import torch.nn as nn

class NN(nn.Module): 
    def __init__(self):
        super().__init__(self)
        self.model = ...

    def forward(self, input_data): 
        return self.model(input_data)
```

## Loss Function

[Loss functions in pytorch](https://docs.pytorch.org/docs/stable/nn.html#loss-functions) are just different criterons that calculate the difference between tensors, sometimes with a weight or other conditional helpers.  As long as they are continious, they can be used to compute a gradient across the entire network during training.

Loss is computed across many batches in large datasets and then reported as cummultive metric, it generally looks something like this: 

```python
import torch.nn.MSELoss as MSELoss
model = NN()
loss_func = MSELoss()
dataset = ...

batch_loss = []
for input_data, target in dataset: # <- iterates across n batches of the dataset
    optimizer.zero_grad() # <- set gradient to its initial condition before computing across a new batch 
    output = model(input_data)
    loss = loss_fn(output, target) # <- use loss.__call__ to get the difference between the expected and predicted
    loss.backward() # <- Compute the gradient from that loss, used in the optimizer

    batch_loss.append(batch_loss.item()) # Only getting the value of the loss, not the gradient, with `.item()`
    
total_batch_loss = sum(batch_loss)/len(batch_loss) # Take the average across the batch to report
```


## Optimizer

The [optimizer](https://docs.pytorch.org/docs/stable/optim.html) takes the specific parameters of the model to optimize, specifics for the type of optimizer being used, and allow you the `step` the optimizer after the loss of the model is computed. 

It generally looks like this: 

```python
from torch.optim import SGD

model = NN()
loss_func = MSELoss()
lr = 0.01 # Some initial condition
optimizer = SGD(
    model.parameters()  # Because model is a torch.nn.Module object, it as the `parameters` function
    lr=lr
)
dataset = ...

for input_data, target in dataset:
    optimizer.zero_grad()
    output = model(input)
    loss = loss_fn(output, target)
    loss.backward()
    optimizer.step() # <- Critical last step, use the backward step of loss to update the model weights

```


In [ ]:
# Generate training and testing data
samples = 2000
train_range = 100
test_range = 200


def func(x):
    return x ** 2


x = np.random.random((samples, 1)) * train_range - (train_range / 2)
y = np.array(list(map(func, x))).reshape(-1, 1)

x_test = np.random.random((int(samples / 2), 1)) * test_range - (test_range / 2)
y_test = np.array(list(map(func, x_test))).reshape(-1, 1)

: 

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(x, y, ".", label="Original Data (y=x^2)")
plt.axvspan(-50, 50, color='green', alpha=0.2, label="Range of training data")
plt.legend()
plt.xlabel('x')

In [ ]:
class SimpleNN(nn.Module): #< - Making a "subclass" of "nn.Module", tells pytorch that this is a neural network
    def __init__(self):
        super().__init__(self) #< - Calling the "constructor" of the parent class, "nn.Module", does setup that lets us call things like the parameters of the model. 

        self.input_layer = nn.Linear(1, 10) #< - A linear layer with 1 input and 10 outputs
        self.hidden_layer = nn.Linear(10, 10) #< - A linear layer
        self.relu = nn.ReLU() #< - A non-linear activation function, it does not have weights, so we don't need a different instance for each time we use it. 
        self.output_layer = nn.Linear(10, 1) #< - A linear layer with 10 inputs and 1 output

    def forward(self, x): # <- This is the forward call, which is used when you call the model on some data, e.g. "model(x)"
        # Use this to say how data is passed from layer to layer
        x = self.input_layer(x) 
        x = self.relu(x)
        x = self.hidden_layer(x)
        x = self.relu(x)
        x = self.output_layer(x)
        return x

summary(SimpleNN(), input_size=(1, 1))

In [ ]:
# Set up the model, loss function, and optimizer
model = SimpleNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# How long we will train the model, how many times to loop through the data
epochs = 100

In [ ]:
# Perform train-test-validation split and convert the numpy data into PyTorch tensors

# Split the existing "train" data further into training and validation sets
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

# Convert the numpy arrays to PyTorch tensors so that we can use them in our model
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

x_val_tensor = torch.tensor(x_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

batch_size = 256
train_dataset = torch.utils.data.TensorDataset(x_train_tensor, y_train_tensor)  # We can use the data.TensorDataset object to make the dataset iterable
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
for epoch in range(epochs):
    model.train() # Tell the model we're in training mode
    # Training step, iterating over the training data in batches
    for batch_x, batch_y in train_dataloader:
        optimizer.zero_grad() # Clear the gradients from the previous step
        outputs = model(batch_x) # Pass the input data through the model to get predictions
        loss = criterion(outputs, batch_y) # Calculate the loss between the predictions and the true values
        loss.backward() # Calculate the gradients of the loss with respect to the model's parameters
        optimizer.step() # Update the model's parameters using the optimizer

    # Validation step
    model.eval() # Tell the model we're in evaluation mode
    with torch.no_grad(): # Disable gradient calculation for validation, as we aren't updating the parameters of the model with the validation data
        val_outputs = model(x_val_tensor) # Pass the validation data through the model to get predictions
        val_loss = criterion(val_outputs, y_val_tensor) # Calculate the loss between the predictions and the true values
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {loss.item():.4f}, Validation Loss: {val_loss.item():.4f}") # Report the validation loss for this epoch

In [ ]:
# Create some predictions on the test data
model.eval() # Tell the model we're in evaluation mode
with torch.no_grad(): # Disable gradient calculation for testing, as we aren't updating the parameters of
    predictions = model(x_test_tensor) # Pass the test data through the model to get predictions
    test_loss = criterion(predictions, y_test_tensor) # Calculate the loss between the predictions and the true values
print(f"Test Loss: {test_loss.item():.4f}") # Report the test loss

In [ ]:
# Plot predictions on test data
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(x_test, y_test, ".", label="Original Data (y=x^2)")
plt.plot(x_test, predictions.numpy(), '.', label="Neural Network")
plt.axvspan(-50, 50, color='green', alpha=0.2, label="Range of training data")
plt.legend()
plt.xlabel('x')
plt.ylabel('prediction')
plt.title('Predictions on Test dataset in {} to {}'.format(-(train_range / 2), (train_range / 2)))
plt.show()

# Test Problem

Some sort of regression task with a physics backing, that we can curl the data down

Present to them the problem and the evalation metrics

# Using neural networks for classification tasks

In [ ]:
# Toy problem

n_features = 20
n_samples = 10000
n_classes = 2

x, y = make_blobs(n_samples=n_samples, n_features=n_features, random_state=42, shuffle=True, return_centers=False, centers=n_classes)
x_test, y_test = make_blobs(n_samples=int(n_samples / 2), n_features=n_features, random_state=42, shuffle=True, return_centers=False, centers=n_classes)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(x[:, 0], x[:, 1], c=y)
plt.xlabel('x_0')
plt.xlabel('x_1')
plt.legend()
plt.title('Original Data (20 features, colored by class)')
plt.show()

In [ ]:
# Defin the model, loss function, and optimizer for the classification problem

class SimpleNN(torch.nn.module):
    def __init__(self):
        super().__init__(self)

        self.input_layer = nn.Linear(??, 10) # <- same as before, but now the input layer has the number of input features as the dataset
        self.hidden_layer = nn.Linear(10, 10)
        self.relu = nn.ReLU()
        self.output_layer = nn.Linear(10, ??) # <- output layer has to output the same number of classes as the dataset has

    def forward(self, x):
        x = self.input_layer(x)
        x = self.relu(x)
        x = self.hidden_layer(x)
        x = self.relu(x)
        x = self.output_layer(x)
        return x
    
model = SimpleNN()
criterion = nn.CrossEntropyLoss() #< - For classification problems, we typically use cross-entropy loss, which combines a softmax activation with a negative log likelihood loss.
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# How do you train the model now? 
# It should look a lot like the training loop we had before

# Using DeepMerge Illustris - Using CNNS


In [ ]:
version = 'pristine'
file_url = 'https://archive.stsci.edu/hlsps/deepmerge/hlsp_deepmerge_hst-jwst_acs-wfc3-nircam_illustris-z2_f814w-f160w-f356w_v1_sim-'+version+'.fits'
hdu = fits.open(download_file(file_url, cache=True, show_progress=True))

In [ ]:
# select random image indices:
# Task: Create a NumPy array consisting of 16 random numbers ranging from 0 to len(hdu[1].data)
# Hint: Refer to the NumPy documentation to complete the line below


example_ids = np.random.choice(len(hdu[1].data),16)
#print(example_ids)

# pull the F160W image (index=1) from the simulated dataset for these selections

examples = [hdu[0].data[j,1,:,:] for j in example_ids]

# initialize your figure
fig1=plt.figure(figsize=(8,8))

# loop through the randomly selected images and plot with labels
for i, image in enumerate(examples):
    plt.subplot(4, 4, i + 1)
    plt.axis("off")

    norm = simple_norm(image, 'log')

    plt.imshow(image, aspect='auto', cmap='viridis', norm=norm)
    plt.title('Merger='+str(bool(hdu[1].data[example_ids[i]][0])))

plt.show()

In [ ]:
X = hdu[0].data #Images
y = hdu[1].data #Labels

X = np.asarray(X).astype('float32')
y = np.asarray(y).astype('float32')

# Task: Split the data into training and validation sets
# Hint: Look for the function train_test_split(???) in scikit-learn

# First split off 30% of the data for validation+testing
X_train, X_split, y_train, y_split = train_test_split(X, y, test_size=0.3,
                                        random_state=42, shuffle=True)

# Divide this subset into validation and testing sets in 1:2 ratio
X_val, X_test, y_val, y_test = train_test_split(X_split, y_split,
                                    test_size=2/3, random_state=42, shuffle=True)

imsize = np.shape(X_train)[2]

X_train = X_train.reshape(-1, imsize, imsize, 3)
X_valid = X_val.reshape(-1, imsize, imsize, 3)
X_test = X_test.reshape(-1, imsize, imsize, 3)

# And convert the data into PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

# Reserved for evalution on at the end
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
# Building a convolusional neural network (CNN) in PyTorch

class GalaxyCNN(nn.Module):
    def __init__(self, imsize):
        super().__init__()
        
        # Conv block 1
        self.conv1 = nn.Conv2d(3, 8, kernel_size=5, stride=1, padding='same')
        self.bn1 = nn.BatchNorm2d(8)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=None)
        self.dropout1 = nn.Dropout(0.5)
        
        # Conv block 2
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, stride=1, padding='same')
        self.bn2 = nn.BatchNorm2d(16)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=None)
        self.dropout2 = nn.Dropout(0.5)
        
        # Conv block 3
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding='same')
        self.bn3 = nn.BatchNorm2d(32)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=None)
        self.dropout3 = nn.Dropout(0.5)
        
        # Fully connected layers
        # Calculate flattened size after 3 pooling layers (each halves spatial dims)
        flattened_size = 32 * (imsize // 8) * (imsize // 8)
        
        self.fc1 = nn.Linear(flattened_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool1(x)
        x = self.dropout1(x)
        
        # Conv block 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.pool2(x)
        x = self.dropout2(x)
        
        # Conv block 3
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.pool3(x)
        x = self.dropout3(x)
        
        # Flatten
        x = torch.flatten(x, 1)
        
        # Fully connected layers
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        
        return x

# Create model instance (assuming imsize is defined from your data)
model = GalaxyCNN(imsize)
summary(model, input_size=(1, 3, imsize, imsize))


In [ ]:
# We need two main things here - a loss function and the optimizer
# The optimizer is a hyperparameter that doesn't need to be changed based on archecture or task, 
# But the loss function is different for classification and regression. 
# A classification loss needed to be able to accept the output of the model (probablity), and the true labels (0, 1)

# Different losses are listed here https://docs.pytorch.org/docs/2.11/nn.html#loss-functions

# The process of training the model looks pretty similar as before, so I won't write out out
# Try to use the example of the training loop we had before, but now with the new model and loss function.

# You will need to track the training and validation loss

In [ ]:
history = {
    "loss" : [], # <- Loss reported by the model
    "val_loss" : [], 
    "accuracy" : [], # <- Accuracy given by sklearn.metrics.accuracy_score
    "val_accuracy" : [] 
} # track these during the training loop

In [ ]:
# Write the training and validation loops


In [ ]:
# plotting from history

loss = history['loss']
val_loss = history['val_loss']
acc = history['accuracy']
val_acc = history['val_accuracy']

epochs = list(range(len(loss)))
figsize=(6,4)
fig, axis1 = plt.subplots(figsize=figsize)
plot1_lacc = axis1.plot(epochs, acc, 'navy', label='accuracy')
plot1_val_lacc = axis1.plot(epochs, val_acc, 'deepskyblue', label="validation accuracy")

plot1_loss = axis1.plot(epochs, loss, 'red', label='loss')
plot1_val_loss = axis1.plot(epochs, val_loss, 'lightsalmon', label="validation loss")


plots = plot1_loss + plot1_val_loss
labs = [l.get_label() for l in plots]
axis1.set_xlabel('Epoch')
axis1.set_ylabel('Loss/Accuracy')
plt.title("Loss/Accuracy History (Pristine Images)")
plt.tight_layout()
axis1.legend(loc='lower right')

In [ ]:
# Now we need to evaluate the trained model on test data
with torch.no_grad(): # Disable gradient calculation for testing, as we aren't updating the parameters of the model with the test data
    test_outputs = model(X_test) # Pass the test data through the model to get predictions
    test_loss = criterion(test_outputs, y_test) # Calculate the loss between the predictions and the true values
print(f"Test Loss: {test_loss.item():.4f}") # Report the test loss

# We can also check a lot of different types of metrics, which all tell us different things about the performance

## Confusion Matrix 

A good option for visualizing the goodness of the predictions is the confusion matrix.

It shows the counts of True Positives (TP), False Positives (FP), True Negatives (TN), and False Negatives (FN).

A perfect classifier would have all predictions along the diagonal (top-left to bottom-right), indicating correct classifications. However, confusion matrices for real-world classifiers may have varied distributions across these categories leading to off-diagonal values.

Understanding the confusion matrix helps in assessing the model's strengths and weaknesses, particularly in identifying where it tends to misclassify instances.

In [ ]:
# measure confusion
labels=[0, 1]
cm = confusion_matrix(y_test, test_outputs, labels=labels)
cm = cm.astype('float') # regular CM
cm_norm = cm / cm.sum(axis=1)[:, np.newaxis] # normalized CM


#plotting
fig = plt.figure()
ax = fig.add_subplot(111)
cax = ax.matshow(cm)
plt.title('Confusion matrix', y=1.08)
fig.colorbar(cax)
ax.set_xticklabels([''] + labels)
ax.set_yticklabels([''] + labels)
plt.xlabel('Predicted')
plt.ylabel('True')
fmt = '.0f'
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], fmt),
        ha="center", va="center",
        color="white" if cm[i, j] < thresh else "black")
plt.show()

# ROC Curve

Another common plot is the Receiver Operating Characteristic (ROC) plot.

It shows the trade-off between TP and FP.

A perfect cassifier has a curve that approaches top left corner, while a classifier that randomly predicts labels has ROC curve close to the diagonal.

Calculating the Area under the ROC curve (AUC) is a good indicator of performance (we aim to be close to AUC = 1).

Read more about the ROC curve and AUC [here](https://keras.rstudio.com/reference/metric_auc.html).

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, test_outputs, pos_label=1)
auc = roc_auc_score(y_test, test_outputs)


figsize=(5,5)
fig, axis1 = plt.subplots(figsize=figsize)
x_onetoone = y_onetoone = [0, 1]
plt.plot(fpr, tpr, 'r-')
plt.plot(x_onetoone, y_onetoone, 'k--',  label="1-1")
plt.legend(loc=0)
plt.title("Receiver Operator Characteristic (ROC)")
plt.xlabel("False Positive (1 - Specificity)")
plt.ylabel("True Positive (Selectivity)")
#Adding text inside a rectangular box by using the keyword 'bbox'
plt.text(0.5, 0.2, "AUC ="+"{0:.2f}".format(auc), fontsize = 22,
         bbox = dict(facecolor = 'red', alpha = 0.5))
plt.tight_layout()